# 02 · SQL analysis

**Goal:** run the ten SQL scripts in `sql/` against `data/inventory.duckdb` and check every output.

The analysis itself lives in SQL, so it can be read, reviewed and reused by anyone who knows SQL. This notebook only runs it and shows the results. The scripts use standard, PostgreSQL-style SQL: CTEs, window functions, `FILTER` clauses and gaps-and-islands.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
con = duckdb.connect(str(ROOT / "data" / "inventory.duckdb"))


def run_script(name):
    """Run every statement in one SQL file and return the result of each SELECT."""
    sql = (ROOT / "sql" / name).read_text()
    results = []
    for statement in con.extract_statements(sql):
        relation = con.execute(statement)
        if relation.description:            # a SELECT returns columns; CREATE does not
            results.append(relation.df())
    return results

## 1. Staging views and quality checks

In [2]:
run_script("01_staging_views.sql")
checks = run_script("02_quality_checks.sql")
labels = ["Q1 size and dates", "Q2 duplicate keys", "Q3 nulls and negatives", "Q4 missing days",
          "Q5 shelf coverage", "Q6 store-closed days", "Q7 prices"]
for label, df in zip(labels, checks):
    print(f"\n{label}")
    print(df.to_string(index=False))


Q1 size and dates
 row_count  items  stores first_date  last_date
  11767600   1612      10 2014-05-24 2016-05-22

Q2 duplicate keys
 duplicate_keys
              0

Q3 nulls and negatives
 null_units  negative_units  null_dates
          0               0           0

Q4 missing days
 item_stores_missing_days
                        0

Q5 shelf coverage
 item_store_days  pct_days_not_listed  sales_without_price  pct_listed_days_zero_sales
        11767600                  1.4                    0                        67.3

Q6 store-closed days
store_id       date  store_units
    CA_1 2014-12-25          0.0
    CA_2 2014-12-25          0.0
    CA_3 2014-12-25          0.0
    CA_4 2014-12-25          0.0
    TX_1 2014-12-25          0.0
    TX_2 2014-12-25          0.0
    TX_3 2014-12-25          0.0
    WI_1 2014-12-25          0.0
    WI_2 2014-12-25          0.0
    WI_3 2014-12-25          0.0
    TX_2 2015-03-24         38.0
    CA_1 2015-12-25          0.0
    CA_2 2015-12-

Q6 confirms the stores were shut on Christmas Day. Without the `store_open` flag, every listed item would show a one-day "stockout" on 25 December.

## 2. Weekly mart, ABC–XYZ, stockouts, network gaps, dead stock, demand drivers and reorder points

In [3]:
for script in [
    "03_mart_item_store_week.sql",
    "04_abc_xyz.sql",
    "05_stockout_runs.sql",
    "06_network_gaps.sql",
    "07_slow_dead_stock.sql",
    "08_demand_drivers.sql",
    "09_reorder_points.sql",
    "10_dashboard_tables.sql",
]:
    run_script(script)

con.sql("""
    SELECT table_name, estimated_size AS rows
    FROM duckdb_tables()
    ORDER BY table_name
""").df()

,table_name,rows
0,abc_xyz,16120
1,abc_xyz_matrix,9
2,calendar,730
3,dash_item_store_summary,16120
4,dash_kpis,1
5,dash_lost_sales_by_store,40
6,dash_slow_dead,332
7,dash_stockout_watchlist,500
8,dash_stockouts_52w,6642
9,dash_weekly_sales,4160


## 3. Results

**Headline numbers** (last 52 weeks)

In [4]:
con.sql("SELECT * FROM dash_kpis").df().T.rename(columns={0: "value"})

,value
revenue_52w,19929074.00
units_52w,4957515.00
item_stores,16120.00
pct_revenue_from_a,80.00
a_item_stores,6168.00
stockout_runs_52w,6642.00
item_stores_with_stockouts,2799.00
est_lost_units_52w,142776.00
est_lost_revenue_52w,359976.00
lost_revenue_pct_of_sales,1.81


**ABC–XYZ matrix:** how revenue and predictability combine

In [5]:
con.sql("SELECT segment, item_stores, revenue_52w, pct_of_revenue, planning_rule FROM abc_xyz_matrix").df()

,segment,item_stores,revenue_52w,pct_of_revenue,planning_rule
0,AX,2857,9916134.66,49.8,Automate replenishment; tight safety stock; re...
1,AY,2913,5311232.91,26.7,Safety stock sized to variability; review weekly
2,AZ,398,721791.96,3.6,Plan with buyers; check daily for stockouts
3,BX,452,363588.51,1.8,Automate replenishment; standard safety stock
4,BY,3129,1937402.84,9.7,Review every two weeks
5,BZ,1310,683723.89,3.4,Order against demand signals; avoid overstock
6,CX,26,10878.94,0.1,Bulk orders; review monthly
7,CY,1315,362495.19,1.8,Keep minimal stock; review monthly
8,CZ,3720,621825.23,3.1,Candidate for delisting or order-on-demand


**Stockout runs by likely cause.** Long gaps are kept out of the lost-sales totals.

In [6]:
con.sql("""
    SELECT likely_cause,
           COUNT(*)                         AS runs,
           ROUND(AVG(run_days), 1)          AS avg_days,
           ROUND(SUM(est_lost_revenue), 0)  AS est_lost_revenue
    FROM stockout_runs
    GROUP BY likely_cause
""").df()

,likely_cause,runs,avg_days,est_lost_revenue
0,Likely stockout,12767,9.2,701343.0
1,Long gap: check for range change,5709,78.0,1852379.0


**The ten most expensive likely stockouts**

In [7]:
con.sql("""
    SELECT store_id, item_id, abc, xyz, run_start, run_days,
           baseline_daily_units, est_lost_units, est_lost_revenue, stores_out_together
    FROM dash_stockout_watchlist
    LIMIT 10
""").df()

,store_id,item_id,abc,xyz,run_start,run_days,baseline_daily_units,est_lost_units,est_lost_revenue,stores_out_together
0,TX_3,HOBBIES_1_354,A,X,2015-09-27,5,19.107,95.5,2195.41,3
1,TX_2,HOBBIES_1_354,A,X,2015-09-26,7,11.375,79.6,1829.78,3
2,CA_3,HOUSEHOLD_1_272,A,Z,2016-01-04,20,7.636,152.7,1365.38,1
3,CA_3,HOUSEHOLD_1_106,A,X,2015-08-28,14,10.679,149.5,1341.02,10
4,CA_3,HOUSEHOLD_1_272,A,Z,2015-07-23,9,13.125,118.1,1177.71,1
5,TX_3,HOUSEHOLD_1_106,A,Y,2015-08-28,16,7.518,120.3,1078.96,10
6,CA_3,HOUSEHOLD_1_205,A,Y,2015-08-31,17,9.304,158.2,939.47,4
7,CA_3,HOUSEHOLD_1_160,A,Y,2015-10-01,18,5.054,91.0,896.50,1
8,TX_2,HOUSEHOLD_1_106,A,X,2015-08-28,16,6.036,96.6,866.25,10
9,CA_1,HOUSEHOLD_1_106,A,X,2015-08-28,16,5.804,92.9,832.93,10


**Store problem or supply problem?** Gaps that hit the same item in three or more stores at once point to the supplier or the distribution centre.

In [8]:
con.sql("""
    SELECT CASE WHEN network_gap THEN '3+ stores out together' ELSE '1-2 stores' END AS gap_type,
           COUNT(*)                                                         AS runs,
           ROUND(SUM(est_lost_revenue), 0)                                  AS est_lost_revenue,
           ROUND(100 * SUM(est_lost_revenue) / SUM(SUM(est_lost_revenue)) OVER (), 1) AS pct_of_lost_revenue
    FROM dash_stockouts_52w
    GROUP BY gap_type
""").df()

,gap_type,runs,est_lost_revenue,pct_of_lost_revenue
0,3+ stores out together,1887,120715.0,33.5
1,1-2 stores,4755,239261.0,66.5


**Range health:** how many lines are slow, dead or delisted

In [9]:
con.sql("""
    SELECT stock_status, COUNT(*) AS item_stores, ROUND(SUM(revenue_52w), 0) AS revenue_52w
    FROM slow_dead_stock
    GROUP BY stock_status
    ORDER BY item_stores DESC
""").df()

,stock_status,item_stores,revenue_52w
0,Active,15788,19873854.0
1,Slow: 2 or fewer units in 13 weeks,305,84351.0
2,Dead: no sales in 26 weeks,27,1492.0


**Demand drivers:** the biggest event uplifts, and SNAP payment days

In [10]:
con.sql("""
    SELECT driver_type, driver, dept_id, days, avg_uplift
    FROM demand_drivers
    WHERE driver_type = 'SNAP payment day'
       OR driver IN (SELECT driver FROM demand_drivers WHERE driver_type = 'Event'
                     GROUP BY driver ORDER BY AVG(avg_uplift) DESC LIMIT 3)
    ORDER BY driver_type, avg_uplift DESC
""").df()

,driver_type,driver,dept_id,days,avg_uplift
0,Event,Cinco De Mayo,HOBBIES_2,6,0.431
1,Event,LaborDay,HOUSEHOLD_1,6,0.262
2,Event,Pesach End,HOBBIES_2,6,0.229
3,Event,LaborDay,HOUSEHOLD_2,6,0.227
4,Event,Cinco De Mayo,HOUSEHOLD_2,6,0.148
5,Event,Pesach End,HOUSEHOLD_2,6,0.114
6,Event,Pesach End,HOBBIES_1,6,0.106
7,Event,Pesach End,HOUSEHOLD_1,6,0.095
8,Event,LaborDay,HOBBIES_2,6,0.077
9,Event,LaborDay,HOBBIES_1,6,0.071


**Reorder points** for the five biggest A-items at a 7-day lead time

In [11]:
con.sql("""
    SELECT r.store_id, r.item_id, r.xyz, r.avg_daily_demand, r.sd_daily_demand,
           r.safety_stock_units, r.reorder_point_units
    FROM reorder_points AS r
    JOIN abc_xyz AS a USING (item_id, store_id)
    WHERE r.lead_time_days = 7
    ORDER BY a.revenue_52w DESC
    LIMIT 5
""").df()

,store_id,item_id,xyz,avg_daily_demand,sd_daily_demand,safety_stock_units,reorder_point_units
0,TX_3,HOBBIES_1_354,X,23.571,8.482,37.0,203.0
1,TX_3,HOBBIES_1_158,X,19.341,6.531,28.5,164.0
2,TX_2,HOBBIES_1_354,X,10.275,4.264,18.6,91.0
3,TX_2,HOBBIES_1_158,X,8.101,3.448,15.1,72.0
4,CA_3,HOBBIES_1_354,X,6.879,3.645,15.9,65.0


In [12]:
con.close()
print("All SQL outputs are saved in data/inventory.duckdb. Next: notebooks/03_insights.ipynb")

All SQL outputs are saved in data/inventory.duckdb. Next: notebooks/03_insights.ipynb
